# ULTRA paper — CLIP explainability notebook

This notebook reproduces and adapts **CLIP explainability** utilities used in the ULTRA paper experiments.

## Credits / dependencies
- **Transformer-MM-Explainability** (Chefer et al.): used for the CLIP model code (imported as `CLIP.clip`).
- **OpenAI CLIP** architecture (ViT-B/32).


In [ ]:
# --- External dependency: Transformer-MM-Explainability (Chefer et al.) ---
# This notebook uses the CLIP implementation vendored in that repo (imported as `CLIP.clip`).

import sys
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/hila-chefer/Transformer-MM-Explainability"
REPO_DIR = Path("external/Transformer-MM-Explainability")

if not REPO_DIR.exists():
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    subprocess.check_call(["git", "clone", REPO_URL, str(REPO_DIR)])

# Make the dependency importable (so `import CLIP.clip as clip` works)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

# Optional: install python deps (uncomment if needed)
# subprocess.check_call([sys.executable, "-m", "pip", "install", "einops", "ftfy", "captum", "opencv-python", "scipy", "tqdm", "matplotlib", "pillow"])


In [ ]:
import os
import sys
import subprocess
from pathlib import Path
from copy import deepcopy

import torch
import numpy as np
import cv2
import matplotlib.pyplot as plt
from PIL import Image

from scipy.spatial.distance import pdist, squareform
from scipy.cluster.hierarchy import linkage, fcluster

import CLIP.clip as clip
from captum.attr import visualization


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# Load CLIP (from the Transformer-MM-Explainability repo)
model, preprocess = clip.load("ViT-B/32", device=device, jit=False)
model.eval();


In [ ]:
def preprocess_image_relevance(relevance, token, res=224,  mode='bicubic'):
    r_image = relevance
    r_image[:, token - 1] = 0
    r_image[:, token - 1] = torch.max(r_image, axis=1)[0]
    r_image = r_image.reshape(-1, 7, 7)
    r_image = r_image.unsqueeze(1)  # Now shape is (batch, 1, 7, 7)

    # Resize each image in the batch to 224x224
    r_image = torch.nn.functional.interpolate(
        r_image, size=res, mode=mode)

    # Remove the channel dimension if not needed, resulting in (8, 224, 224)
    r_image = r_image.squeeze(1)
    r_image = r_image.reshape(-1, res*res)
    return r_image


def map_to_sequential_indices(tensor):
    # Find unique values in the tensor and sort them
    unique_vals = torch.unique(tensor)

    # Create a mapping from unique values to sequential indices
    mapping = {val.item(): idx for idx, val in enumerate(unique_vals)}

    # Apply the mapping to the tensor
    mapped_tensor = tensor.clone()
    for val, idx in mapping.items():
        mapped_tensor[tensor == val] = idx

    return mapped_tensor

In [ ]:
def unsup_seg(image, model, device, layer=13, distance_threshold=0.25, res=7, linkage_method='average', dist='cosine', normalized=True):
    seg_torch, seg_numpy = [], []
    size = model.visual.conv1.stride[0]
    input_res = model.visual.input_resolution
    num_token = int(input_res/size)
    for token in range(1, num_token):

        r_image = interpret_token(image, layer, token, model, device)
        r_image = preprocess_image_relevance(r_image, token, res)
        seg_numpy.append(r_image.squeeze().cpu().numpy())
        seg_torch.append(r_image.cpu())

    seg_numpy = np.array(seg_numpy).transpose(1, 0, 2)
    seg_torch = torch.stack(seg_torch).transpose(1, 0)

    segs = []
    for j, seg_num in enumerate(seg_numpy):

        cosine_distances = pdist(seg_num, metric=dist)
        # if hierarchical :
        # Perform hierarchical clustering using scipy
        linkage_matrix = linkage(cosine_distances, linkage_method)
        labels = fcluster(linkage_matrix, t=distance_threshold,
                          criterion='distance')

        # Find unique clusters
        unique_labels = np.unique(labels)
        # Initialize clustering dictionary
        clustered = {label: [] for label in unique_labels}

        # Assign each segment to its cluster
        for i, label in enumerate(labels):
            clustered[label].append(seg_torch[j, i])

        # Sum each cluster tensor
        clss = []
        if normalized:

            for cluster in clustered:
                summed_cluster = torch.stack(
                    clustered[cluster], dim=0).sum(dim=0)

                # Normalize the summed cluster tensor between 0 and 5
                min_val = summed_cluster.min()
                max_val = summed_cluster.max()
                normalized_cluster = 5 * \
                    (summed_cluster - min_val) / (max_val - min_val)

                clss.append(normalized_cluster)

        else:
            for cluster in clustered:
                clss.append(torch.stack(clustered[cluster], dim=0).sum(dim=0))

        clss = torch.stack(clss)
        seg = torch.argmax(clss, dim=0)

        seg_numpy = deepcopy(seg.reshape(res, res).detach().cpu().numpy())
        seg_reshape = cv2.resize(seg_numpy.astype(
            np.float32), (224, 224), interpolation=cv2.INTER_NEAREST)
        segs.append(torch.tensor(seg_reshape))

    return torch.stack(segs)


def interpret_token(image, layer, token_id, model, device, texts=None):
    attn_blocks = [i for i in model.visual.transformer.resblocks.children()]

    image = image.to(device)
    image = image.half() if str(device).startswith('cuda') else image.float()
    x = model.visual.conv1(image)
    # shape = [*, width, grid ** 2]   [B, channel, H*W]
    x = x.reshape(x.shape[0], x.shape[1], -1)
    # shape = [*, grid ** 2, width]       [B, H*W, channel]
    x = x.permute(0, 2, 1)
    x = torch.cat([model.visual.class_embedding.to(x.dtype) + torch.zeros(x.shape[0], 1, x.shape[-1],
                  dtype=x.dtype, device=x.device), x], dim=1)  # shape = [*, grid ** 2 + 1, width]
    x = x + model.visual.positional_embedding.to(x.dtype)
    x = model.visual.ln_pre(x)
    x = x.permute(1, 0, 2)  # NLD -> LND    [H*W, B, channel]
    for i in range(len(attn_blocks)):
        x = attn_blocks[i](x)
        if i+2 == layer:
            token = x[token_id]
    x = x.permute(1, 0, 2)
    x = x = model.visual.ln_post(x[:, 0, :])
    final = x @ model.visual.proj

    batch_size = image.shape[0]
    token_sum = torch.norm(token)
    model.zero_grad()

    num_tokens = attn_blocks[0].attn_probs.shape[-1]

    R = torch.eye(num_tokens, num_tokens,
                  dtype=attn_blocks[0].attn_probs.dtype).to(device)
    R = R.unsqueeze(0).expand(batch_size, num_tokens, num_tokens)
    normal = 0
    for i, blk in enumerate(attn_blocks):
        if i+1 >= layer:
            continue
        normal += 1
        grad = torch.autograd.grad(
            token_sum, [blk.attn_probs], retain_graph=True, allow_unused=True)
        grad = grad[0].detach()
        cam = blk.attn_probs.detach()
        cam = cam.reshape(-1, cam.shape[-1], cam.shape[-1])
        grad = grad.reshape(-1, grad.shape[-1], grad.shape[-1])
        cam = grad * cam
        cam = cam.reshape(batch_size, -1, cam.shape[-1], cam.shape[-1])
        cam = cam.clamp(min=0).mean(dim=1)
        R = R + torch.bmm(cam, R)

    image_relevance = R[:, token_id, 1:]
    return image_relevance  # [batch* patch * patch]

In [ ]:
def token_to_patch_xy(token_id: int, num_patches: int, patch_size: int):
    """Map a patch token id (1..num_patches**2) to pixel-space (x, y) top-left coordinates."""
    if token_id <= 0:
        return None
    idx = token_id - 1
    col = idx % num_patches
    row = idx // num_patches
    return (col * patch_size, row * patch_size)


In [ ]:
# --- Input image ---
# Put your image under `assets/` (recommended) or point this path to your data.
img_name = "pic1"

assets_path = Path("assets") / f"{img_name}.jpg"
fallback_path = REPO_DIR / "CLIP" / f"{img_name}.jpg"  # sample image shipped by dependency

img_path = assets_path if assets_path.exists() else fallback_path
if not img_path.exists():
    raise FileNotFoundError(f"Could not find {assets_path} (or fallback {fallback_path}).")

img = preprocess(Image.open(img_path)).unsqueeze(0).to(device)

# Quick preview
plt.figure(figsize=(4,4))
plt.imshow(Image.open(img_path))
plt.axis("off")
plt.title(str(img_path))
plt.show()


In [ ]:
def show_tensor_img(image):
    plt.figure()
    image = image.permute(1, 2, 0).data.cpu().numpy()
    image = (image - image.min()) / (image.max() - image.min())
    plt.imshow(image)
    plt.axis('off')
    from pathlib import Path
    out_dir = Path("outputs")
    out_dir.mkdir(parents=True, exist_ok=True)
    file_name = out_dir / f"{img_name}.pdf"
    plt.savefig(str(file_name), format="pdf", bbox_inches="tight")

In [ ]:
def image_relevance_on_ax(ax, image_relevance, image, layer, token_index, title=None, patch_pos=None, patch_size=0,save=False):
    def show_cam_on_image(img, mask):
        heatmap = cv2.applyColorMap(np.uint8(255 * mask), cv2.COLORMAP_JET)
        heatmap = np.float32(heatmap) / 255
        cam = heatmap + np.float32(img)
        cam = cam / np.max(cam)
        return cam

    dim = int(image_relevance.numel() ** 0.5)
    image_relevance = image_relevance.reshape(1, 1, dim, dim)
    image_relevance = torch.nn.functional.interpolate(image_relevance, size=224, mode='bilinear')
    image_relevance = image_relevance.reshape(224, 224).detach().cpu().numpy()
    image_relevance = (image_relevance - image_relevance.min()) / (image_relevance.max() - image_relevance.min())

    image = image[0].permute(1, 2, 0).data.cpu().numpy()
    image = (image - image.min()) / (image.max() - image.min())
    vis = show_cam_on_image(image, image_relevance)
    vis = np.uint8(255 * vis)
    vis = cv2.cvtColor(np.array(vis), cv2.COLOR_RGB2BGR)

    # Draw the patch rectangle if patch_pos is provided
    if patch_pos and patch_size > 0:
        startj, starti = patch_pos
        # Change the rectangle color to purple (R:128, G:0, B:128)
        cv2.rectangle(vis, (startj, starti), (startj + patch_size, starti + patch_size), (128, 0, 128), 2)

    ax.imshow(vis)
    ax.axis('off')

    # Add title and token index if provided
    if title:
        ax.set_title(title)

    # Annotate with token index
    # ax.text(10, 20, f'Token: {token_index}', color='white', fontsize=12, weight='bold', backgroundcolor='black')

    # Save figure to PDF
    if save:
      file_name = f"layer_{layer}_token_{token_index}.pdf"
      ax.figure.savefig(file_name, format='pdf', bbox_inches='tight')
      print(f"Figure saved as {file_name}")


In [ ]:
def save_token_img(image_relevance, image, layer, token_index,patch_pos,patch_size):
    def show_cam_on_image(img, mask):
          heatmap = cv2.applyColorMap(np.uint8(255 * mask), cv2.COLORMAP_JET)
          heatmap = np.float32(heatmap) / 255
          cam = heatmap + np.float32(img)
          cam = cam / np.max(cam)
          return cam

    dim = int(image_relevance.numel() ** 0.5)
    image_relevance = image_relevance.reshape(1, 1, dim, dim)
    image_relevance = torch.nn.functional.interpolate(image_relevance, size=224, mode='bilinear')
    image_relevance = image_relevance.reshape(224, 224).detach().cpu().numpy()
    image_relevance = (image_relevance - image_relevance.min()) / (image_relevance.max() - image_relevance.min())

    image = image[0].permute(1, 2, 0).data.cpu().numpy()
    image = (image - image.min()) / (image.max() - image.min())
    vis = show_cam_on_image(image, image_relevance)
    vis = np.uint8(255 * vis)
    vis = cv2.cvtColor(np.array(vis), cv2.COLOR_RGB2BGR)

    # Draw the patch rectangle if patch_pos is provided
    if patch_pos and patch_size > 0:
        startj, starti = patch_pos
        # Change the rectangle color to purple (R:128, G:0, B:128)
        cv2.rectangle(vis, (startj, starti), (startj + patch_size, starti + patch_size), (128, 0, 128), 2)

    from pathlib import Path

    out_dir = Path("outputs") / str(img_name)
    out_dir.mkdir(parents=True, exist_ok=True)
    file_name = out_dir / f"layer_{layer}_token_{token_index}.pdf"

    plt.figure()

    plt.imshow(vis)
    plt.axis('off')
    plt.savefig(str(file_name), format='pdf', bbox_inches='tight',dpi=150)
    # print(f"Figure saved as {file_name}")

In [ ]:
# --- Example: explain a single token ---
layer = 13       # visual transformer block index (see paper/code for conventions)
token_id = 11    # 1..(7*7)=49 for ViT-B/32 patch tokens

patch_size = model.visual.conv1.stride[0]
num_patches = model.visual.input_resolution // patch_size
patch_xy = token_to_patch_xy(token_id, num_patches=num_patches, patch_size=patch_size)

R = interpret_token(img, layer=layer, token_id=token_id, model=model, device=device)
# preprocess_image_relevance modifies the tensor in-place, so pass a clone
R_img = preprocess_image_relevance(R.clone(), token=token_id, res=224)

fig, ax = plt.subplots(1, 1, figsize=(4, 4))
image_relevance_on_ax(ax, R, img, layer=layer, token_index=token_id, patch_pos=patch_xy, patch_size=patch_size)
plt.show()


In [ ]:
# --- Save a PDF overlay to outputs/ (optional) ---
# This writes: outputs/<img_name>/layer_<...>_token_<...>.pdf
save_token_img(R, img, layer=layer, token_index=token_id, patch_pos=patch_xy, patch_size=patch_size)


In [ ]:
# --- Optional: visualize many tokens/layers (can be slow) ---
# Tip: start small (e.g., tokens=range(1, 13), layers=range(2, 7))
from tqdm import tqdm

def visualize_grid(tokens, layers, figsize_per_cell=(2,2)):
    fig, ax = plt.subplots(len(tokens), len(layers), figsize=(figsize_per_cell[0]*len(layers), figsize_per_cell[1]*len(tokens)))
    if len(tokens) == 1 and len(layers) == 1:
        ax = np.array([[ax]])
    elif len(tokens) == 1:
        ax = ax[None, :]
    elif len(layers) == 1:
        ax = ax[:, None]

    patch_size = model.visual.conv1.stride[0]
    num_patches = model.visual.input_resolution // patch_size

    for i, tok in enumerate(tqdm(tokens)):
        patch_xy = token_to_patch_xy(tok, num_patches, patch_size)
        for j, lyr in enumerate(layers):
            R = interpret_token(img, layer=lyr, token_id=tok, model=model, device=device)
            image_relevance_on_ax(ax[i, j], R, img, layer=lyr, token_index=tok, patch_pos=patch_xy, patch_size=patch_size)
            ax[i, j].set_title(f"L{lyr} T{tok}", fontsize=8)

    plt.tight_layout()
    plt.show()

# Example (small):
# visualize_grid(tokens=list(range(1, 7)), layers=list(range(2, 6)))
